# bench-aitextdetect — AI-Generated Text Detection Benchmark

This notebook benchmarks **LLM fine-tuning** (QLoRA) and classical ML baselines on
the task of distinguishing **human-written** from **machine-generated** text.

**Dataset**: [MAGE](https://huggingface.co/datasets/yaful/MAGE) (ACL 2024) —
437 k samples across multiple domains and generators (GPT-3.5-turbo, GPT-4,
LLaMA, etc.).  Binary labels: `0 = machine-generated`, `1 = human-written`.

**Models benchmarked**:

| Category | Model | Size | Method |
|----------|-------|------|--------|
| Baselines | TF-IDF + LogReg / RF / XGBoost / LightGBM | — | Classical ML on n-gram features |
| Encoder (QLoRA) | [ModernBERT-base](https://huggingface.co/answerdotai/ModernBERT-base) | 149 M | 4-bit NF4, LoRA r=16, 8192-token context |
| Encoder (QLoRA) | [ModernBERT-large](https://huggingface.co/answerdotai/ModernBERT-large) | 395 M | 4-bit NF4, LoRA r=16, 8192-token context |
| Causal LM (QLoRA) | [Qwen3.5-4B-Base](https://huggingface.co/Qwen/Qwen3.5-4B-Base) | 4 B | 4-bit NF4, LoRA r=16 |
| Causal LM (QLoRA) | [Gemma-4-E4B](https://huggingface.co/google/gemma-4-E4B) | 4 B eff. | 4-bit NF4, LoRA r=16 |

**Evaluation**:
- In-domain: MAGE test set + two OOD splits (`test_ood_set_gpt`, `test_ood_set_gpt_para`)
- Cross-dataset: [artem9k/ai-text-detection-pile](https://huggingface.co/datasets/artem9k/ai-text-detection-pile) — multi-generator robustness

The QLoRA profile below is tuned for a **32 GB CUDA GPU**. Classical baselines run on CPU.

## 1. Environment Setup

In [ ]:
import os, subprocess, sys
from pathlib import Path

cwd = Path.cwd()
repo_root = cwd.parent if cwd.name == "ml_pipeline" else cwd
if (repo_root / "ml_pipeline").is_dir() and (repo_root / "requirements.txt").is_file():
    os.chdir(repo_root)
    print(f"Using local checkout: {repo_root}")
else:
    subprocess.run(["rm", "-rf", "bench_research_ml_project"], check=True)
    subprocess.run(["git", "clone", "https://github.com/oremaz/bench_research_ml_project"], check=True)
    os.chdir("bench_research_ml_project")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.51.3", "peft==0.18.0", "bitsandbytes==0.47.0", "accelerate==1.12.0"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets", "rich"], check=True)

In [ ]:
from pathlib import Path
import os

if Path.cwd().name != "ml_pipeline":
    os.chdir("ml_pipeline")
print(f"Working directory: {Path.cwd()}")

## 2. Imports and Deterministic Utilities

In [ ]:
import os, random, logging
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from pipelines_torch.models import (
    CLASSIFICATION_MODEL_REGISTRY,
    HuggingFaceQLoRAWrapper,
)
from pipelines_torch.benchmark import BenchmarkRunner
from pipelines_torch.base import GeneralPipeline, GeneralPipelineSklearn, SimplePredictor
from utils.metrics import METRIC_REGISTRY
from utils.utils import save_model, save_metrics, model_exists, load_model_by_name

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Download and Explore the MAGE Dataset

[MAGE](https://huggingface.co/datasets/yaful/MAGE) (ACL 2024) provides:
- **train** (319 k), **valid** (57 k), **test** (61 k) — standard splits
- **test_ood_set_gpt** — out-of-distribution GPT-generated texts
- **test_ood_set_gpt_para** — paraphrased GPT texts (harder to detect)

Columns: `text`, `label` (0 = machine, 1 = human).

In [ ]:
from datasets import load_dataset

ds = load_dataset("yaful/MAGE")
print(ds)

# Preview
for split_name in ds:
    print(f"\n--- {split_name} ({len(ds[split_name]):,} rows) ---")
    print(ds[split_name][0])

In [ ]:
# Convert to DataFrames for easier manipulation
df_train = ds["train"].to_pandas()
df_val = ds["validation"].to_pandas()
df_test = ds["test"].to_pandas()

# OOD test sets (may have different column names)
ood_splits = {k: ds[k].to_pandas() for k in ds if k not in ("train", "validation", "test")}
print(f"OOD splits available: {list(ood_splits.keys())}")

# Basic stats
print(f"\nTrain: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,}")
print(f"\nTrain label distribution:\n{df_train['label'].value_counts()}")
print(f"\nTest label distribution:\n{df_test['label'].value_counts()}")

# Text length distribution
df_train["text_len"] = df_train["text"].str.len()
print(f"\nText length stats (train):\n{df_train['text_len'].describe()}")

In [ ]:
# Subsample for tractable training (full dataset is 319k — too slow for QLoRA grid search)
# Use 10k train, 2k val, full test sets
TRAIN_SIZE = 10_000
VAL_SIZE = 2_000

df_train_sub = df_train.groupby("label", group_keys=False).apply(
    lambda g: g.sample(n=TRAIN_SIZE // 2, random_state=SEED)
).reset_index(drop=True)

df_val_sub = df_val.groupby("label", group_keys=False).apply(
    lambda g: g.sample(n=min(VAL_SIZE // 2, len(g)), random_state=SEED)
).reset_index(drop=True)

print(f"Subsampled train: {len(df_train_sub):,} | val: {len(df_val_sub):,}")
print(f"Train labels:\n{df_train_sub['label'].value_counts()}")

X_train = df_train_sub["text"].tolist()
y_train = df_train_sub["label"].values

X_val = df_val_sub["text"].tolist()
y_val = df_val_sub["label"].values

X_test = df_test["text"].tolist()
y_test = df_test["label"].values

## 4. Classical ML Baselines (TF-IDF)

TF-IDF vectorisation + four classifiers from the repo registry.
These run on CPU and serve as strong baselines — TF-IDF frequently
performs surprisingly well on stylistic detection tasks.

In [ ]:
# Fit TF-IDF on training texts
tfidf = TfidfVectorizer(max_features=50_000, sublinear_tf=True, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train).toarray().astype(np.float32)
X_val_tfidf = tfidf.transform(X_val).toarray().astype(np.float32)
X_test_tfidf = tfidf.transform(X_test).toarray().astype(np.float32)

print(f"TF-IDF features: {X_train_tfidf.shape[1]:,}")

In [ ]:
from pipelines_torch.models import (
    SklearnRandomForestClassifierWrapper,
    XGBoostClassifierWrapper,
    LightGBMClassifierWrapper,
)

baseline_configs = [
    {
        "name": "tfidf_logreg",
        "class": lambda input_dim, num_classes=2: LogisticRegression(
            max_iter=1000, C=1.0, solver="lbfgs", random_state=SEED
        ),
        "params": {"input_dim": X_train_tfidf.shape[1], "num_classes": 2},
    },
    {
        "name": "tfidf_random_forest",
        "class": SklearnRandomForestClassifierWrapper,
        "params": {"input_dim": X_train_tfidf.shape[1], "num_classes": 2},
    },
    {
        "name": "tfidf_xgboost",
        "class": XGBoostClassifierWrapper,
        "params": {"input_dim": X_train_tfidf.shape[1], "num_classes": 2},
    },
    {
        "name": "tfidf_lightgbm",
        "class": LightGBMClassifierWrapper,
        "params": {"input_dim": X_train_tfidf.shape[1], "num_classes": 2},
    },
]

# Combine train+val for sklearn (no early stopping needed)
X_baseline = np.vstack([X_train_tfidf, X_val_tfidf])
y_baseline = np.concatenate([y_train, y_val])

runner_baseline = BenchmarkRunner(
    model_configs=baseline_configs,
    augmentations=[None],
    task_type="classification",
    device="cpu",
    epochs=1,  # sklearn models ignore this
    batch_size=32,
    early_stopping=None,
    use_class_weights=True,
    use_kfold=True,
    k_folds=3,
    use_wandb=True,
    path_start="bench_aitextdetect",
    random_state=SEED,
)
runner_baseline.run(X_baseline, y_baseline)

## 5. QLoRA Fine-Tuning

This notebook uses a **32 GB VRAM** QLoRA profile. Four models are benchmarked
with the repo's `HuggingFaceQLoRAWrapper` (4-bit NF4 quantization, LoRA rank 16),
each in its own `BenchmarkRunner`:

| Model | Type | Params | Batch size | Grad accum | Effective batch | Max tokens | Epochs | Notes |
|-------|------|--------|-----------:|-----------:|----------------:|-----------:|-------:|-------|
| ModernBERT-base | Encoder | 149 M | 16 | 1 | 16 | 2048 | 2 | More throughput while retaining longer-document signal |
| ModernBERT-large | Encoder | 395 M | 8 | 2 | 16 | 2048 | 2 | Strong encoder setting that still leaves memory headroom |
| Qwen3.5-4B-Base | Causal LM | 4 B | 4 | 4 | 16 | 1024 | 2 | Causal activations dominate memory, so context is capped lower |
| Gemma-4-E4B | Causal LM | 4 B eff. | 4 | 4 | 16 | 1024 | 2 | 32 GB can carry E4B at 4-bit; use E2B only as an OOM fallback |

> **32 GB GPU memory profile**
> - The quantized model weights fit comfortably; activation memory from batch size and sequence
>   length is the real limiter during fine-tuning.
> - Encoders get a 2048-token training context because ModernBERT's attention pattern is cheaper
>   for long documents.
> - Causal/hybrid LMs stay at 1024 tokens with gradient checkpointing; raise to 2048 only after
>   confirming your local run has memory headroom.
> - Gemma-4-**E4B** replaces E2B for this profile. If you hit OOM, switch `hf_model` back to
>   `google/gemma-4-E2B` and keep the rest of the config unchanged.

**Wrapper changes for causal / hybrid LLMs** (in `pipelines_torch/models.py`):
- Auto-detects causal vs encoder architecture -> sets `padding_side="left"` for causal LMs
- Sets `pad_token = eos_token` when missing; propagates `pad_token_id` into `model.config`
- Auto-detects head name (`"classifier"` vs `"score"`) for `modules_to_save`
- Falls back from `target_modules="all-linear"` to explicit attention+MLP modules for MoE/SSM layers
- Uses `gradient_checkpointing_kwargs={"use_reentrant": False}` for causal/hybrid LMs
- Accepts a wrapper-level `max_seq_length` so each notebook config controls token budget


In [ ]:
QLORA_VRAM_PROFILE = "32gb"

QLORA_CONFIGS = {
    "qlora_modernbert_base": {
        "hf_model": "answerdotai/ModernBERT-base",
        "batch_size": 16,
        "epochs": 2,
        "lr": 1e-4,
        "gradient_accumulation_steps": 1,
        "max_seq_length": 2048,
    },
    "qlora_modernbert_large": {
        "hf_model": "answerdotai/ModernBERT-large",
        "batch_size": 8,
        "epochs": 2,
        "lr": 1e-4,
        "gradient_accumulation_steps": 2,
        "max_seq_length": 2048,
    },
    "qlora_qwen35_4b": {
        "hf_model": "Qwen/Qwen3.5-4B-Base",
        "batch_size": 4,
        "epochs": 2,
        "lr": 5e-5,
        "gradient_accumulation_steps": 4,
        "max_seq_length": 1024,
    },
    "qlora_gemma4_e4b": {
        "hf_model": "google/gemma-3-4b-it",  # gemma-4-E4B config unsupported by AutoModelForSequenceClassification
        "batch_size": 4,
        "epochs": 2,
        "lr": 5e-5,
        "gradient_accumulation_steps": 4,
        "max_seq_length": 1024,
    },
}

ALL_QLORA_MODELS = {name: cfg["hf_model"] for name, cfg in QLORA_CONFIGS.items()}
print(f"QLoRA VRAM profile: {QLORA_VRAM_PROFILE}")
print("QLoRA models:", list(ALL_QLORA_MODELS.keys()))


In [ ]:
def run_qlora_model(name, cfg, X, y, path_start="bench_aitextdetect"):
    """Run a single QLoRA model through BenchmarkRunner with its own GPU-budget settings."""
    hf_model = cfg["hf_model"]
    grad_accum = cfg.get("gradient_accumulation_steps", 2)
    max_seq_length = cfg.get("max_seq_length", 512)

    model_config = [{
        "name": name,
        "class": lambda model_name=hf_model, ga=grad_accum, msl=max_seq_length, **kw: HuggingFaceQLoRAWrapper(
            model_name=model_name,
            num_labels=2,
            task_type="classification",
            device=DEVICE,
            gradient_accumulation_steps=ga,
            max_seq_length=msl,
        ),
        "params": {},
    }]

    runner = BenchmarkRunner(
        model_configs=model_config,
        augmentations=[None],
        task_type="classification",
        device=DEVICE,
        epochs={name: cfg["epochs"]},
        batch_size=cfg["batch_size"],
        early_stopping=None,
        use_class_weights=False,
        use_kfold=False,
        learning_rate=cfg["lr"],
        use_wandb=True,
        path_start=path_start,
        random_state=SEED,
    )
    runner.run(np.array(X), y)
    print(f"  Done: {name}")


for model_name, model_cfg in QLORA_CONFIGS.items():
    effective_batch = model_cfg["batch_size"] * model_cfg["gradient_accumulation_steps"]
    print(f"\n{'='*60}")
    print(f"Training: {model_name}  ({model_cfg['hf_model']})")
    print(
        f"  batch={model_cfg['batch_size']}  epochs={model_cfg['epochs']}"
        f"  grad_accum={model_cfg['gradient_accumulation_steps']}  effective_batch={effective_batch}"
        f"  max_seq={model_cfg['max_seq_length']}  lr={model_cfg['lr']}"
    )
    print('='*60)
    run_qlora_model(model_name, model_cfg, X_train, y_train)


## 6. Evaluate All Models on Test Set

Score every trained model on the held-out MAGE test split using the shared
metric registry (accuracy, F1, precision, recall, ROC-AUC, PR-AUC).

In [ ]:
METRIC_MAP = {
    "accuracy": "accuracy",
    "f1_macro": "f1",
    "precision_macro": "precision",
    "recall_macro": "recall",
    "roc_auc": "roc_auc",
    "pr_auc": "pr_auc",
}


def evaluate_sklearn_models(model_names, X_eval, y_eval, tfidf_vectorizer):
    """Evaluate sklearn-based models that need TF-IDF features."""
    records = []
    X_tfidf = tfidf_vectorizer.transform(X_eval).toarray().astype(np.float32)

    for name, cfg in zip(model_names, baseline_configs):
        try:
            model_class = cfg["class"]
            model = load_model_by_name(
                model_class, name, cfg["params"],
                path_start="bench_aitextdetect"
            )
        except FileNotFoundError:
            print(f"  Skipping {name}: checkpoint not found")
            continue

        if hasattr(model, "predict_proba"):
            probs = model.predict_proba(X_tfidf)
            if probs.ndim == 1:
                probs = np.column_stack([1 - probs, probs])
        else:
            preds = model.predict(X_tfidf)
            probs = np.zeros((len(preds), 2))
            probs[np.arange(len(preds)), preds.astype(int)] = 1.0

        scores = {
            label: float(METRIC_REGISTRY[key](y_eval, probs))
            for label, key in METRIC_MAP.items()
        }
        records.append({"model": name, **scores})
        print(f"  {name}: acc={scores['accuracy']:.4f} f1={scores['f1_macro']:.4f}")

    return pd.DataFrame.from_records(records)


def evaluate_qlora_models(qlora_model_dict, X_eval, y_eval):
    """Evaluate QLoRA fine-tuned models.

    qlora_model_dict: {name: hf_model_id}, e.g. ALL_QLORA_MODELS
    """
    records = []

    for name, hf_model in qlora_model_dict.items():
        try:
            model_class = lambda model_name=hf_model, **kw: HuggingFaceQLoRAWrapper(
                model_name=model_name, num_labels=2,
                task_type="classification", device=DEVICE,
            )
            model = load_model_by_name(
                model_class, name, {},
                path_start="bench_aitextdetect"
            )
        except FileNotFoundError:
            print(f"  Skipping {name}: checkpoint not found")
            continue

        probs = model.predict_proba(X_eval)
        scores = {
            label: float(METRIC_REGISTRY[key](y_eval, probs))
            for label, key in METRIC_MAP.items()
        }
        records.append({"model": name, **scores})
        print(f"  {name}: acc={scores['accuracy']:.4f} f1={scores['f1_macro']:.4f}")

    return pd.DataFrame.from_records(records)

In [ ]:
print("=== Baseline (TF-IDF) models on MAGE test set ===")
baseline_names = [cfg["name"] for cfg in baseline_configs]
df_baseline_results = evaluate_sklearn_models(baseline_names, X_test, y_test, tfidf)

print("\n=== QLoRA models on MAGE test set ===")
df_qlora_results = evaluate_qlora_models(ALL_QLORA_MODELS, X_test, y_test)

df_all_results = pd.concat([df_baseline_results, df_qlora_results], ignore_index=True)
df_all_results = df_all_results.sort_values("f1_macro", ascending=False).reset_index(drop=True)
df_all_results

## 7. Out-of-Distribution (OOD) Generalisation

The MAGE dataset ships with two OOD test sets to evaluate robustness:
- **test_ood_set_gpt** — texts from GPT models not seen during training
- **test_ood_set_gpt_para** — GPT texts that have been paraphrased (hardest setting)

This mirrors the cross-dataset evaluation done in `bench-imai` for images.

In [ ]:
ood_results = {}

for ood_name, ood_df in ood_splits.items():
    print(f"\n=== OOD: {ood_name} ({len(ood_df):,} samples) ===")
    X_ood = ood_df["text"].tolist()
    y_ood = ood_df["label"].values
    print(f"Label distribution: {dict(zip(*np.unique(y_ood, return_counts=True)))}")

    print("\n--- Baselines ---")
    df_bl = evaluate_sklearn_models(baseline_names, X_ood, y_ood, tfidf)

    print("\n--- QLoRA ---")
    df_ql = evaluate_qlora_models(ALL_QLORA_MODELS, X_ood, y_ood)

    ood_combined = pd.concat([df_bl, df_ql], ignore_index=True)
    ood_combined["dataset"] = ood_name
    ood_results[ood_name] = ood_combined

if ood_results:
    df_ood_all = pd.concat(ood_results.values(), ignore_index=True)
    df_ood_all

## 8. Cross-Dataset Evaluation — AI Text Detection Pile

[artem9k/ai-text-detection-pile](https://huggingface.co/datasets/artem9k/ai-text-detection-pile)
is a large-scale, multi-generator corpus covering GPT-2, GPT-3, GPT-J, and ChatGPT outputs
mixed with human text.  It is a **different distribution** from MAGE — different domains,
different generation methods — so performance here measures genuine cross-dataset generalisation.

All models are evaluated **zero-shot** (no retraining): we apply the MAGE-trained checkpoints
directly to this dataset.  We subsample to 5 k samples to keep inference time tractable.

In [ ]:
PILE_EVAL_SIZE = 5_000

ds_pile = load_dataset("artem9k/ai-text-detection-pile", split="train")
df_pile = ds_pile.to_pandas()

print(f"AI Text Detection Pile — {len(df_pile):,} rows")
print("Columns:", df_pile.columns.tolist())
print(df_pile.head(3))

In [ ]:
# Normalise columns: the current pile schema is ['source', 'id', 'text'], where
# 'source' is a string ('human' or 'ai') rather than a boolean/int flag.
# Remap to the same 'text' / 'label' convention as MAGE (0=machine, 1=human)
text_col = next(c for c in df_pile.columns if c.lower() == "text")
label_col = next(
    (c for c in df_pile.columns if c.lower() in ("is_generated", "label", "generated", "ai")),
    None
)
source_col = next((c for c in df_pile.columns if c.lower() == "source"), None)

df_pile = df_pile.rename(columns={text_col: "text"})

if label_col and label_col != "label":
    # is_generated=1 means machine, so label = 1 - is_generated (human=1 like MAGE)
    df_pile["label"] = 1 - df_pile[label_col].astype(int)
elif label_col == "label":
    df_pile["label"] = df_pile["label"].astype(int)
elif source_col:
    # source names the generator ('human' vs a model name); human=1 like MAGE
    df_pile["label"] = (df_pile[source_col].astype(str).str.lower() == "human").astype(int)
else:
    raise ValueError(f"Cannot find label column in: {df_pile.columns.tolist()}")

print(f"Label distribution:\n{df_pile['label'].value_counts()}")

# Balanced subsample
df_pile_sub = df_pile.groupby("label", group_keys=False).apply(
    lambda g: g.sample(n=min(PILE_EVAL_SIZE // 2, len(g)), random_state=SEED)
).reset_index(drop=True)

X_pile = df_pile_sub["text"].tolist()
y_pile = df_pile_sub["label"].values
print(f"\nEval set: {len(df_pile_sub):,} samples | labels: {dict(zip(*np.unique(y_pile, return_counts=True)))}")

In [ ]:
print("=== Cross-dataset eval: AI Text Detection Pile ===")

print("\n--- Baselines ---")
df_pile_baseline = evaluate_sklearn_models(baseline_names, X_pile, y_pile, tfidf)

print("\n--- QLoRA ---")
df_pile_qlora = evaluate_qlora_models(ALL_QLORA_MODELS, X_pile, y_pile)

df_pile_results = pd.concat([df_pile_baseline, df_pile_qlora], ignore_index=True)
df_pile_results = df_pile_results.sort_values("f1_macro", ascending=False).reset_index(drop=True)
df_pile_results

## 9. Results Visualisation

In [ ]:
# --- Bar chart: In-domain test performance ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax = axes[0]
df_plot = df_all_results.sort_values("accuracy", ascending=True)
colors = ["#2196F3" if "qlora" in m else "#FF9800" for m in df_plot["model"]]
ax.barh(df_plot["model"], df_plot["accuracy"], color=colors)
ax.set_xlabel("Accuracy")
ax.set_title("MAGE Test Set — Accuracy")
ax.set_xlim(0.5, 1.0)
for i, v in enumerate(df_plot["accuracy"]):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)

# F1 Macro
ax = axes[1]
df_plot = df_all_results.sort_values("f1_macro", ascending=True)
colors = ["#2196F3" if "qlora" in m else "#FF9800" for m in df_plot["model"]]
ax.barh(df_plot["model"], df_plot["f1_macro"], color=colors)
ax.set_xlabel("F1 (macro)")
ax.set_title("MAGE Test Set — F1 Macro")
ax.set_xlim(0.5, 1.0)
for i, v in enumerate(df_plot["f1_macro"]):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("results/bench_aitextdetect_indomain.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/bench_aitextdetect_indomain.png")

In [ ]:
# --- Grouped bar chart: In-domain vs OOD generalisation ---
if ood_results:
    # Build a combined view: in-domain + each OOD split
    df_all_results_copy = df_all_results.copy()
    df_all_results_copy["dataset"] = "test (in-domain)"
    df_combined = pd.concat([df_all_results_copy, df_ood_all], ignore_index=True)

    models = df_combined["model"].unique()
    datasets = df_combined["dataset"].unique()
    n_datasets = len(datasets)

    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(len(models))
    width = 0.8 / n_datasets
    palette = ["#4CAF50", "#2196F3", "#FF5722", "#9C27B0"]

    for i, ds_name in enumerate(datasets):
        subset = df_combined[df_combined["dataset"] == ds_name]
        # Align by model order
        vals = []
        for m in models:
            row = subset[subset["model"] == m]
            vals.append(row["f1_macro"].values[0] if len(row) else 0)
        ax.bar(x + i * width, vals, width, label=ds_name, color=palette[i % len(palette)])

    ax.set_ylabel("F1 (macro)")
    ax.set_title("In-Domain vs OOD Generalisation — F1 Macro")
    ax.set_xticks(x + width * (n_datasets - 1) / 2)
    ax.set_xticklabels(models, rotation=30, ha="right")
    ax.legend()
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.savefig("results/bench_aitextdetect_ood.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: results/bench_aitextdetect_ood.png")

In [ ]:
# --- MAGE in-domain vs Cross-dataset (AI Text Detection Pile) ---
df_mage = df_all_results.copy()
df_mage["dataset"] = "MAGE (in-domain)"
df_cross = df_pile_results.copy()
df_cross["dataset"] = "AI-Text-Pile (cross-dataset)"

df_cross_compare = pd.concat([df_mage, df_cross], ignore_index=True)

models_cross = df_cross_compare["model"].unique()
datasets_cross = ["MAGE (in-domain)", "AI-Text-Pile (cross-dataset)"]
x = np.arange(len(models_cross))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
palette = ["#4CAF50", "#E91E63"]

for i, ds_name in enumerate(datasets_cross):
    subset = df_cross_compare[df_cross_compare["dataset"] == ds_name]
    vals = []
    for m in models_cross:
        row = subset[subset["model"] == m]
        vals.append(row["f1_macro"].values[0] if len(row) else 0)
    bars = ax.bar(x + i * width, vals, width, label=ds_name, color=palette[i])
    for bar, v in zip(bars, vals):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                    f"{v:.2f}", ha="center", va="bottom", fontsize=8)

ax.set_ylabel("F1 (macro)")
ax.set_title("MAGE vs AI-Text-Detection-Pile — Cross-Dataset Generalisation")
ax.set_xticks(x + width / 2)
ax.set_xticklabels(models_cross, rotation=30, ha="right")
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig("results/bench_aitextdetect_cross_dataset.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/bench_aitextdetect_cross_dataset.png")

## 10. Save Checkpoints Archive

In [ ]:
import tarfile
from IPython.display import FileLink

results_path = "results/bench_aitextdetect"
if os.path.exists(results_path):
    archive_path = "results/bench_aitextdetect_checkpoints.tar.gz"
    with tarfile.open(archive_path, "w:gz") as tar:
        tar.add(results_path, arcname="bench_aitextdetect")
    print(f"Archive created: {archive_path}")
    FileLink(archive_path)
else:
    print("No results directory found — run training cells first.")

## 11. Analysis and Lessons Learned

### Overview

This benchmark evaluates **8 models** (4 classical ML + 4 QLoRA fine-tuned LLMs) on
AI-generated text detection using the MAGE dataset (ACL 2024), with cross-dataset
evaluation on the AI Text Detection Pile.

### Models Compared

| Model | Type | Parameters | QLoRA config | Training |
|-------|------|-----------|-------------|---------|
| TF-IDF + LogReg | Classical | ~50 k features | — | LogReg on bigram TF-IDF |
| TF-IDF + RF | Classical | 100 trees | — | Random Forest |
| TF-IDF + XGBoost | Classical | Gradient boosted | — | XGBoost |
| TF-IDF + LightGBM | Classical | Gradient boosted | — | LightGBM |
| ModernBERT-base | Encoder | 149 M + LoRA | 4-bit NF4, r=16, batch=16, grad_accum=1, max_seq=2048 | 2 epochs, lr=1e-4 |
| ModernBERT-large | Encoder | 395 M + LoRA | 4-bit NF4, r=16, batch=8, grad_accum=2, max_seq=2048 | 2 epochs, lr=1e-4 |
| Qwen3.5-4B-Base | Causal LM | 4 B + LoRA | 4-bit NF4, r=16, batch=4, grad_accum=4, max_seq=1024 | 2 epochs, lr=5e-5 |
| Gemma-4-E4B | Causal LM | 4 B eff. + LoRA | 4-bit NF4, r=16, batch=4, grad_accum=4, max_seq=1024 | 2 epochs, lr=5e-5 |

---

### Key Questions Addressed

**1. Do QLoRA-finetuned LLMs outperform TF-IDF baselines?**
- TF-IDF captures surface-level n-gram artifacts that differ between human and AI text (repetition,
  vocabulary distribution, sentence-length uniformity). These are strong signals on in-domain data.
- QLoRA models capture deeper pragmatic/semantic cues: reasoning structure, discourse coherence,
  hedging patterns. They should outperform particularly on OOD and paraphrased texts.

**2. ModernBERT-base vs ModernBERT-large?**
- Both are encoders with identical architecture — large is simply wider.
- For classification, the encoder advantage (bidirectional attention, 8192-token context)
  should scale with capacity: large should outperform base, particularly on longer texts.
- At ~3 GB at 4-bit, ModernBERT-large is far cheaper than the 4B causal LMs.

**3. Qwen3.5-4B-Base vs Gemma-4-E4B?**
- **Qwen3.5-4B-Base**: un-instruction-tuned, clean fine-tune signal, hybrid MoE gives broader
  world knowledge; 262K native context. The 32 GB profile uses max_seq=1024 and effective batch 16.
- **Gemma-4-E4B**: larger effective Gemma setting enabled by 32 GB VRAM. It uses the same
  max_seq=1024 and effective batch 16 budget as Qwen, with E2B kept as the OOM fallback.

**4. Cross-dataset generalisation (AI Text Detection Pile)?**
- The Pile covers GPT-2/3/J/ChatGPT — different generators and domains from MAGE.
- TF-IDF models may collapse here: their n-gram vocabulary is tuned to MAGE's domain mix.
- Encoder models (ModernBERT) should transfer better than causal LMs.

---

### Expected Findings

| Setting | Expected winner | Rationale |
|---------|----------------|-----------|
| MAGE in-domain | Qwen, Gemma-E4B, or ModernBERT-large | Stronger 32 GB QLoRA profile with effective batch 16 |
| MAGE OOD (GPT variants) | ModernBERT-large | Bidirectional attention, more capacity |
| MAGE OOD (paraphrased) | TF-IDF trees | Paraphrasing disrupts LLM-learned patterns |
| AI-Text-Pile (cross-dataset) | ModernBERT-large | Better transfer than causal LMs |
| GPU efficiency | ModernBERT-base | Smallest model, encoder, fastest inference |

---

### Next Steps

1. **Full-scale training** — use all 319 k MAGE training samples for production-grade models
2. **Adversarial robustness** — test against paraphrasing attacks, style transfer, backtranslation